# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets with their @id and field @id
print('Available record sets:')
for record_set in metadata.record_sets:
    print(f"- RecordSet @id: {record_set.id}, name: {record_set.name}")
    print(f"  Fields (@id):")
    for field in record_set.fields:
        print(f"    - {field.id}: {field.name} ({field.data_type})")

# Preview several records from the first record set (by @id)
print('\n---\nSample records from the first record set:')
if metadata.record_sets:
    first_record_set = metadata.record_sets[0].id
    for idx, record in enumerate(dataset.records(record_set=first_record_set)):
        print(record)
        if idx == 2:
            break
else:
    print('No record sets found in this dataset.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
record_sets = [rs.id for rs in metadata.record_sets]
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

if len(record_sets) > 0:
    main_record_set_id = record_sets[0]
    print(f"Columns in main record set '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print('No record sets available to extract.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# EDA on a numeric field in the main record set
import numpy as np

# Automatically select a numeric field, looking for typical mass spectrometry columns
numeric_field = None
main_fields = [field for field in metadata.record_sets[0].fields]
df = dataframes[main_record_set_id]

# List of possible numeric columns, adjust as found in overview cell
common_numeric_terms = ['abundance', 'MW', 'sequence_coverage', 'peptide_count', 'score', 'pI', 'norm', 'intensity']
for field in main_fields:
    # Try to find a column matching a known numeric type
    col = field.name
    if any(term in col.lower() for term in common_numeric_terms):
        # Verify the field exists in DataFrame and is numeric
        if col in df.columns and np.issubdtype(df[col].dtype, np.number):
            numeric_field = col
            numeric_field_id = field.id
            break

if numeric_field is None:
    # If not found, pick the first float column (if any)
    for field in main_fields:
        col = field.name
        if col in df.columns and np.issubdtype(df[col].dtype, np.number):
            numeric_field = col
            numeric_field_id = field.id
            break

if numeric_field is None:
    print('No numeric field found for EDA. Please inspect DataFrame columns for available fields.')
else:
    print(f"Using numeric field: '{numeric_field}' (@id: {numeric_field_id})")
    threshold = df[numeric_field].mean() if not df[numeric_field].isnull().all() else 0
    
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with '{numeric_field}' > {threshold:.2f} (mean): {len(filtered_df)} rows.")

    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized '{numeric_field}' for filtered records (first 5 rows):")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    # Try to find a categorical/group field for groupby/aggregation
    group_field = None
    possible_group = ['sample', 'type', 'accession', 'description', 'group', 'condition']
    for field in main_fields:
        col = field.name
        if col in df.columns and any(term in col.lower() for term in possible_group):
            group_field = col
            group_field_id = field.id
            break

    if group_field:
        grouped = (
            filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        )
        print(f"\nGrouped mean '{numeric_field}' by '{group_field}' (@id: {group_field_id}):")
        print(grouped.head())
    else:
        print('No suitable group field found for grouping. Skipping group aggregation.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
    
    # If group_field found, visualize mean by group
    if group_field:
        plt.figure(figsize=(10,4))
        group_means = (
            df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
        )
        sns.barplot(x=group_means.index, y=group_means.values)
        plt.title(f"Top 10 '{group_field}' by mean '{numeric_field}'")
        plt.ylabel(f'Mean {numeric_field}')
        plt.xlabel(group_field)
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print('No numeric field selected for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset was loaded successfully using the `mlcroissant` library, and several record sets and fields were listed by their `@id`s.
- Data from the main record set was extracted, numeric fields were analyzed, filtered, normalized, and visualized.
- This notebook provides a foundation for further biomarker research, comparative protein abundance analysis, and downstream machine learning tasks on this mass spectrometry dataset.
- For reproducibility and accurate field references, all fields and record sets were referenced by their unique `@id` as required by the Croissant standard.